In [1]:
from datetime import datetime
from functools import partial
import os
import time

from absl import app, flags, logging
import click
import cv2
import imageio
import jax
import jax.numpy as jnp
import numpy as np

from octo.model.octo_model import OctoModel
from octo.utils.gym_wrappers import HistoryWrapper, TemporalEnsembleWrapper
from octo.utils.train_callbacks import supply_rng


2024-11-13 13:48:27.339950: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-11-13 13:48:27.339976: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-11-13 13:48:27.341012: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-13 13:48:27.972202: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import sys
sys.path.append("../")
from magpie.gripper import Gripper
from magpie import ur5 as ur5
import magpie.realsense_wrapper as real
from magpie.perception import pcd
from magpie.prompt_planner.prompts import mp_prompt_tc_vision as mptc
from PIL import Image

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
[Open3D INFO] Resetting default logger to print to terminal.


In [3]:
hmpth = "/home/will/workspace/models"
MODEL_CKPT_DICT = {
    "dp": f"{hmpth}/DG.PTH",
    "dp_nf": f"{hmpth}/DGNF.PTH",
    "dp_go": f"{hmpth}/DGGO.PTH",
    "dp_go_nf": f"{hmpth}/DGGONF.PTH",
    "octo_sm_ft": f"{hmpth}/octo_sm_dg",
    "octo_ba_ft": f"{hmpth}/octo_ba_dg",
    "octo_sm_go": f"{hmpth}/octo_sm_dggo",
    "octo": ""
}


In [4]:
model = OctoModel.load_pretrained(MODEL_CKPT_DICT["octo_sm_ft"], 9999)

tokens.shape=(1, 16, 768)
pad_mask_dict.keys()=dict_keys(['language_instruction'])
keys=('language_instruction',)
key='language_instruction'
pad_mask_dict[key].shape=(1,)
tokens.shape=(1, 2, 256, 512)
pad_mask_dict.keys()=dict_keys(['image_primary', 'image_wrist', 'proprio', 'timestep'])
keys=['image_primary']
key='image_primary'
pad_mask_dict[key].shape=(1, 2)
key='image_wrist'
pad_mask_dict[key].shape=(1, 2)
key='proprio'
pad_mask_dict[key].shape=(1, 2)
key='timestep'
pad_mask_dict[key].shape=(1, 2)


In [5]:
policy_fn = supply_rng(
    partial(
        model.sample_actions,
        unnormalization_statistics=model.dataset_statistics["action"],
    ),
)


In [6]:
instruction = "pick up the blue block"
task = model.create_tasks(texts=instruction)

In [7]:
OBJECT_NAME = "blue block"
CONFIG = {}
CONFIG['vla'] = "octo_sm_dg"
SERVO_PORT = "/dev/ttyACM0"
GRIPPER = None
ROBOT_IP = "192.168.0.4"
VLA_ROBOT = ur5.UR5_Interface(ROBOT_IP)
GRIPPER = Gripper(SERVO_PORT)
CAMERA_SERIAL_INFO = real.poll_devices()
WRIST_CAMERA = real.RealSense(fps=5, w=640, h=480, device_name="D405")
WRIST_CAMERA.initConnection(device_serial=CAMERA_SERIAL_INFO['D405'])
WORKSPACE_CAMERA = real.RealSense(zMax=5, fps=6, w=640, h=480, device_name="D435")
WORKSPACE_CAMERA.initConnection(device_serial=CAMERA_SERIAL_INFO['D435'])


Succeeded to open the port
Succeeded to change the baudrate


In [8]:
lang_task = f"grasp {OBJECT_NAME} and return home"
sensors = {
    "robot": VLA_ROBOT,
    "gripper": GRIPPER,
    "wrist_camera": WRIST_CAMERA,
    "workspace_camera": WORKSPACE_CAMERA,
    "language_instruction": lang_task,
}

In [9]:
def get_observation(sensors, obs_queue, last_obs, first_obs=True, cfg="octo_sm_dg"):
    obs = {}
    
    # try:
    #     sensors["robot"].start()
    # except Exception as e:
    #     print(f"Error starting robot: {e}")

    def process_image(image, size=(128, 128), order=(2, 0, 1)):
        # reshape image from 640x480x3 to 3x480x640 (H, W, C) --> (C, H, W)
        image = np.array(Image.fromarray(image).resize(size))
        image = np.transpose(image, order)
        return image

    wksp_size = (256, 256) if "octo" in cfg else (128, 128)

    joints = sensors["robot"].get_joint_angles()
    tcp = np.array(sensors["robot"].recv.getActualTCPPose())
    gripper_pos = np.array([sensors["gripper"].get_aperture()])/100.0
    applied_force = np.array([sensors["gripper"].applied_force])/100.0
    contact_force = np.array([sensors["gripper"].recorded_contact_force])
    action_blocked = np.array([False])
    obs["proprio"] = np.concatenate((joints, tcp, gripper_pos, applied_force, contact_force, action_blocked))
    obs["image_primary"] = process_image(sensors["workspace_camera"].take_image_blocking(), size=wksp_size, order=(0, 1, 2))
    obs["image_wrist"] = process_image(sensors["wrist_camera"].take_image_blocking(), order=(0, 1, 2))
    # obs["pad_mask_dict/timestep"] = False
    # obs["pad_mask_dict/proprio"] = True
    # obs["pad_mask_dict/image_primary"] = True
    # obs["pad_mask_dict/image_wrist"] = True
    # obs["task_completed"] = False
    # obs["timestep"] = time.time()
    obs["timestep_pad_mask"] = False if first_obs else True
    # for k in obs:
    #     print(f"{k}: {obs[k].shape}")

    # try:
    #     sensors["robot"].stop()
    # except Exception as e:
    #     print(f"Error stopping robot: {e}")

    # window=2 so observations with shape (N, ...) become (2, N)
    if first_obs:
        # double the observation
        obs_queue.append({k: np.array([v, v]) for k, v in obs.items()})
    else:
        # take the last_obs and append the new observation to it
        obs_queue.append({k: np.array([last_obs[k], v]) for k, v in obs.items()})

    return obs, obs_queue

In [10]:
OBS_QUEUE = []
LAST_OBS = None

In [12]:
sensors["robot"].start()

Succeeded to open the port
Succeeded to change the baudrate


In [13]:
obs, OBS_QUEUE = get_observation(sensors, OBS_QUEUE, LAST_OBS, first_obs=True, cfg=CONFIG['vla'])
LAST_OBS = obs
batch_window_obs = jax.tree_map(lambda x: x[None], OBS_QUEUE[-1])

In [14]:
for k in OBS_QUEUE[-1]:
    print(f"{k}: {OBS_QUEUE[-1][k].shape}")

proprio: (2, 16)
image_primary: (2, 256, 256, 3)
image_wrist: (2, 128, 128, 3)
timestep_pad_mask: (2,)


In [15]:
for k in batch_window_obs:
    print(f"{k}: {batch_window_obs[k].shape}")

image_primary: (1, 2, 256, 256, 3)
image_wrist: (1, 2, 128, 128, 3)
proprio: (1, 2, 16)
timestep_pad_mask: (1, 2)


In [16]:
action = model.sample_actions(
    batch_window_obs, 
    task,
    unnormalization_statistics=model.dataset_statistics["action"]
    )


tokens.shape=(1, 16, 768)
pad_mask_dict.keys()=dict_keys(['language_instruction'])
keys=('language_instruction',)
key='language_instruction'
pad_mask_dict[key].shape=(22,)


ValueError: Incompatible shapes for broadcasting: (22, 1) and requested shape (1, 16)

In [17]:
# write batch_window_obs to file
np.save("batch_window_obs.npy", batch_window_obs)

In [18]:
# load batch_window_obs from file
bwo = np.load("batch_window_obs.npy", allow_pickle=True).item()

In [19]:
bwo

{'image_primary': array([[[[[ 66,  64,  50],
           [ 66,  62,  45],
           [ 71,  61,  45],
           ...,
           [ 92,  87,  81],
           [ 89,  88,  79],
           [ 87,  87,  74]],
 
          [[ 67,  64,  49],
           [ 66,  62,  46],
           [ 70,  62,  45],
           ...,
           [ 90,  88,  79],
           [ 88,  88,  77],
           [ 85,  86,  74]],
 
          [[ 68,  64,  48],
           [ 68,  61,  47],
           [ 68,  61,  46],
           ...,
           [ 89,  87,  78],
           [ 88,  87,  77],
           [ 87,  87,  76]],
 
          ...,
 
          [[ 17, 165, 253],
           [ 15, 168, 253],
           [ 18, 171, 254],
           ...,
           [131, 124, 105],
           [133, 125, 106],
           [135, 126, 107]],
 
          [[ 15, 164, 249],
           [ 12, 164, 250],
           [ 13, 167, 252],
           ...,
           [131, 124, 105],
           [132, 125, 106],
           [134, 127, 108]],
 
          [[ 15, 165, 246],
   